In [4]:
from datetime import date

import pandas as pd
from sqlalchemy import and_, func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons (applicants and inventors)
    TLS207_PERS_APPLN,          # person <-> application links
    TLS209_APPLN_IPC,           # IPC classification
    TLS224_APPLN_CPC,           # CPC classification
    TLS231_INPADOC_LEGAL_EVENT, # legal events
)

In [5]:
%pip install -q pandas numpy sqlalchemy sentence-transformers

import hashlib
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer


Note: you may need to restart the kernel to use updated packages.


In [ ]:
ENV = "TEST"                  # PROD per l'estrazione finale
YEARS = (2000, 2024)          # anni di deposito: scelta modificabile

SBERT_MODEL = "graziasveva93/sbert-adapt-ub"

TAU_FILTER = 0.45             # esempi iniziali, da calibrare
TAU_ENRICH = 0.60
USE_ENRICHMENT = False        # tested and dropped: Step0/validation/enrichment_decision.csv

THRESHOLDS_FROZEN = False     # True dopo la calibrazione
RANDOM_STATE = 42

DRONE_DESCRIPTION = """
Unmanned aerial vehicles, remotely piloted aircraft and autonomous flying
drones. Technologies specifically designed for their airframe, propulsion,
flight control, navigation, communications, swarm coordination, dedicated
charging infrastructure, launch, landing, recovery and mission equipment.
The invention must provide a concrete technical solution for an unmanned
aircraft. Generic technologies with only an incidental drone application,
ground-based anti-drone systems, manned aircraft and exclusively terrestrial
or underwater vehicles are outside the sector.
""".strip()

Il settore comprende i velivoli senza pilota a bordo, comandati a distanza o autonomi, e le tecnologie sviluppate specificamente per la loro costruzione, il funzionamento e lo svolgimento delle missioni. 
Consideriamo i velivoli indipendentemente dall’impiego finale, senza limitazioni iniziali per paese o impresa.

| Area | Esempi di invenzioni pertinenti |
|---|---|
| **Velivolo** | Struttura, ali, rotori, configurazioni e propulsione specifiche per UAV |
| **Navigazione e controllo** | Pianificazione del volo, stabilizzazione, decollo, atterraggio ed evitamento degli ostacoli |
| **Comunicazioni e coordinamento** | Collegamento con la stazione di controllo, coordinamento di flotte e sciami |
| **Energia e infrastrutture dedicate** | Ricarica, sostituzione delle batterie, stazioni di lancio e recupero progettate per droni |
| **Attrezzature per le missioni** | Sistemi di consegna, ispezione, rilevamento o altre funzioni adattate specificamente agli UAV |

Cosa escludere
- Droni subacquei, di superficie e robot terrestri.
- Velivoli con pilota a bordo.
- Batterie, telecamere, algoritmi e comunicazioni generici, quando il drone è soltanto uno dei possibili utilizzi.
- Sistemi anti-drone installati a terra, trattandoli inizialmente come settore adiacente. Un drone progettato per intercettarne altri rientrerebbe invece nel perimetro.
- Articoli che riguardano esclusivamente l’aspetto estetico del prodotto.
La regola più importante per i casi dubbi
L’invenzione introduce una soluzione specifica per un drone o descrive una tecnologia generica che potrebbe essere utilizzata anche su un drone?


In [7]:
patstat = PatstatClient(env=ENV) # default: env="TEST"; use PatstatClient(env="PROD") for the full database
db = patstat.orm()  # SQLAlchemy session used by all ORM queries below

In [8]:
#"remote-controlled aircraft","remote controlled aircraft",

DRONE_KEYWORDS = (
    # "unmanned/unpiloted" + aircraft/aerial (both papers)
    "unmanned aerial", "unmanned aircraft", "unmanned air vehicle",
    "unpiloted aerial", "unpiloted aircraft", "unpiloted air vehicle",
    # remote* within 1 word of aircraft (DTI)
    "remotely piloted aircraft",
    "micro air vehicle",                                    # DTI
    # rotor types (Gorry)
    "quadrotor", "quadrocopter", "quadcopter", "multicopter", "multirotor", "multi-rotor",
    "hexacopter", "octocopter",
    " uav", "uav ",
    "drone",                                                # contested: keep only if hand-labelling supports it
)
#these are  possible keywords found in scientific paper

In [9]:
DRONE_KEYWORDS = (
    # Core UAV terminology
    "unmanned aerial vehicle",
    "unmanned aerial system",
    "unmanned aircraft",
    "unmanned aircraft system",
    "unmanned air vehicle",
    "uncrewed aircraft",
    "remotely piloted aircraft",

    # UAV configurations
    "quadcopter",
    "quadrotor",
    "quadrocopter",
    "multicopter",
    "hexacopter",
    "octocopter",
    "micro air vehicle",
    "fixed-wing uav",
    "vtol uav",

    # UAV-specific technologies
    "uav navigation",
    "uav flight control",
    "uav communication",
    "uav swarm",
    "uav propulsion",

    # Broad term
    "drone",
)

In [10]:
DRONE_IPC_CODES = (
    "B64U",
)

In [11]:
DRONE_CPC_CODES = (
    "B64U",
    "B64C39/024",
    "G08G5/57",
)

In [12]:
def keyword_condition(terms):
    """OR-condition: any term appears in the title or in the abstract (case-insensitive)."""
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(
        *(title.like(f"%{t.lower()}%") for t in terms),
        *(abstract.like(f"%{t.lower()}%") for t in terms),
    )


def symbol_prefix_condition(symbol_column, codes):
    """OR-condition: the blank-stripped classification symbol starts with one of the codes."""
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


# Application-level fields retrieved by the search queries.
# These fields describe each application; they do not change the search criteria.
APPLN_COLUMNS = (
    TLS201_APPLN.appln_id,              # Unique application identifier
    TLS201_APPLN.docdb_family_id,       # Links applications belonging to the same DOCDB family
    TLS201_APPLN.appln_auth,            # Filing authority or patent office
    TLS201_APPLN.appln_nr,              # Application number
    TLS201_APPLN.appln_kind,            # Application kind code
    TLS201_APPLN.appln_filing_date,     # Filing date of this application
    TLS201_APPLN.earliest_filing_year,  # Earliest filing year associated with this application
    TLS201_APPLN.granted,               # Grant indicator: "Y" means granted
)

#3 TYPES OF QUERIES

#PROF BASELINE, MODIFIES:
#in each -> .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))

def query_by_keywords(*columns):
    """Applications whose title or abstract contains an Drone keyword."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(keyword_condition(DRONE_KEYWORDS))
    )


def query_by_ipc(*columns):
    """Applications with at least one IPC symbol in the Drone list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))
        .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
        .filter(symbol_prefix_condition(TLS209_APPLN_IPC.ipc_class_symbol, DRONE_IPC_CODES))
    )


def query_by_cpc(*columns):
    """Applications with at least one CPC symbol in the Drone list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .filter(TLS201_APPLN.appln_filing_year.between(*YEARS))
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, DRONE_CPC_CODES))
    )

In [13]:
# COMBINE THE SEARCH RESULTS

# Select application IDs found by at least one search strategy:
# keywords OR IPC codes OR CPC codes.
# UNION removes duplicate IDs across the three searches.

appln_id = TLS201_APPLN.appln_id.label("appln_id")

drone_ids = (
    query_by_keywords(appln_id)
    .union(
        query_by_ipc(appln_id),
        query_by_cpc(appln_id),
    )
    .subquery()
)

#each query (keyword,CPC,IPD) retrieve only appln_id foreach application

#with this function we retrieve all the others attributes
def query_details(ids_table):
    return (
        db.query(
            *APPLN_COLUMNS,
            TLS202_APPLN_TITLE.appln_title,
            TLS203_APPLN_ABSTR.appln_abstract,
        )
        .select_from(TLS201_APPLN)
        .join(
            ids_table,
            TLS201_APPLN.appln_id == ids_table.c.appln_id,
        )
        .outerjoin(
            TLS202_APPLN_TITLE,
            and_(
                TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
                func.lower(func.trim(TLS202_APPLN_TITLE.appln_title_lg)) == "en",
            ),
        )
        .outerjoin(
            TLS203_APPLN_ABSTR,
            and_(
                TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
                func.lower(func.trim(TLS203_APPLN_ABSTR.appln_abstract_lg)) == "en",
            ),
        )
        .distinct()
    )

# Prepare the DataFrame with one row per patent application (appln_id).
# Replace missing titles and abstracts with empty strings.
# If multiple rows exist for the same application, keep the longest abstract.
def one_application(df):
    columns = [c.key for c in APPLN_COLUMNS]
    columns += ["appln_title", "appln_abstract"]
    df = df.reindex(columns=columns).copy()

    for column in ["appln_title", "appln_abstract"]:
        df[column] = df[column].fillna("").astype(str).str.strip()

    df["_abstract_length"] = df.appln_abstract.str.len()

    return (
        df.sort_values(
            ["appln_id", "_abstract_length", "appln_abstract", "appln_title"],
            ascending=[True, False, True, True],
        )
        .drop_duplicates("appln_id")
        .drop(columns="_abstract_length")
        .reset_index(drop=True)
    )

q_drone = query_details(drone_ids)
df_drone = one_application(patstat.df(q_drone))

df_drone.head(10)

,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,appln_title,appln_abstract
0,25546,39689224,EP,08157886,A,2008-06-09,2007,Y,Airborne manipulator unmanned aerial vehicle (...,A manipulator arm system (1) on a ducted air-f...
1,352060,33524069,EP,04017945,A,2004-07-29,2003,N,A flying machine,The flying machine (1) comprises a supporting ...
2,6750614,26877844,CN,01807474,A,2001-02-12,2000,N,Remotely piloted aircraft,
3,14034049,11074674,DE,60119462,T,2001-09-10,2000,Y,,
4,14969349,36002220,DE,102004063205,A,2004-12-23,2004,Y,"Aircraft for transporting persons, has rotors/...",The aircraft has rotors/propellers (1) with sh...
5,14971374,36650390,DE,102005002278,A,2005-01-18,2005,N,Method of monitoring land or terrain and deter...,The method involves the aircraft (10) flying a...
6,16419618,39768825,EP,08008008,A,2008-04-25,2007,Y,Enhanced passive coherent location techniques ...,A system and technique is described which has ...
7,19115737,37460436,FR,0651048,A,2006-03-27,2006,Y,Hydrodynamic or streamlined profile for formin...,The profile (1) has an envelope mounted on an ...
8,23690715,32040150,WO,2005000520,W,2005-02-14,2004,N,HIGH ALTITUDE WIND ENERGY FLYING CONVERTIER,Power generating apparatus (10) including a ca...
9,38402196,33535776,WO,2004018183,W,2004-12-07,2004,N,ROTATING DUCT TYPE ROTOR WITH SHROUD,[PROBLEMS] To provide a rotor with a shroud op...


In [15]:
#SECTION 2 : SBERT AS FILTER

# Create a new dataframe by selecting applications with a non-empty English abstract for SBERT.
df_sbert = df_drone[
    df_drone["appln_abstract"].fillna("").str.strip().ne("")
].copy()

print("Applications with an abstract:", len(df_sbert))
print("Applications without an abstract:", len(df_drone) - len(df_sbert))

Applications with an abstract: 3000
Applications without an abstract: 232


In [16]:
model = SentenceTransformer(SBERT_MODEL)

#Converts each abstract in embedding with SBERT, the output is a matrix where every row is one abstract
abstract_embeddings = model.encode(
    df_sbert["appln_abstract"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

#Same for our field description
drone_embedding = model.encode(
    DRONE_DESCRIPTION, 
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Embedding matrix:", abstract_embeddings.shape) # (n_patents, embedding_dim)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Embedding matrix: (3000, 384)


In [17]:
#add a new column foreach application, wich is the similarity between its abstract and the field description
df_sbert["sbert_similarity"] = (
    abstract_embeddings @ drone_embedding  # cosine similarity
)

#sort application based on that similarity
df_sbert[
    ["appln_id", "appln_title", "sbert_similarity"]
].sort_values("sbert_similarity", ascending=False).head(10)

,appln_id,appln_title,sbert_similarity
261,420109751,Mobile unmanned aerial vehicle infrastructure ...,0.972148
733,476005433,Unmanned aerial vehicles,0.969736
2038,529853207,Persistent aerial reconnaissance and communica...,0.964739
1939,525554232,UAS display system and method,0.964722
1951,526197129,UAS DISPLAY SYSTEM AND METHOD,0.964722
2148,535047433,Intelligent combined unmanned aerial vehicle s...,0.963156
502,450284337,METHODS AND APPARATUS FOR UNMANNED AERIAL VEHI...,0.961404
1071,489851321,Autonomous-flight unmanned aerial vehicle syst...,0.958258
2377,545392934,Flight route management method,0.955307
228,414254046,UNMANNED DEVICE INTERACTION METHODS AND SYSTEMS,0.953943


In [18]:
# Select applications whose similarity reaches the filtering threshold.
df_filtered = df_sbert[
    df_sbert["sbert_similarity"] >= TAU_FILTER
].copy()

print("Applications evaluated with SBERT:", len(df_sbert))
print("Applications retained:", len(df_filtered))
print("Applications excluded by the filter:", len(df_sbert) - len(df_filtered))

df_filtered[
    ["appln_id", "appln_title", "sbert_similarity", "appln_abstract"]
].sort_values("sbert_similarity", ascending=False).head(10)

# Scartati dal filtro iniziale
df_rejected = df_sbert[
    df_sbert["sbert_similarity"] < TAU_FILTER
].copy()


Applications evaluated with SBERT: 3000
Applications retained: 2489
Applications excluded by the filter: 511


In [19]:
#SECTION 3 : SBERT for enrichment

# Define broad IPC and CPC prefixes for the enrichment search.
# These codes define a candidate pool, not confirmed drone patents.
EXTERNAL_IPC_CODES = ("B64", "G05D1/", "G08G5/")
EXTERNAL_CPC_CODES = ("B64", "G05D1/", "G08G5/")

# Build a query that selects application IDs matching the given code prefixes.
# Use the same filing-year range as the initial search.
def query_pool(table, symbol_column, codes):
    return (
        db.query(TLS201_APPLN.appln_id.label("appln_id"))
        .select_from(TLS201_APPLN)
        .join(
            table,
            TLS201_APPLN.appln_id == table.appln_id,
        )
        .filter(
            TLS201_APPLN.appln_filing_year.between(*YEARS)
        )
        .filter(
            symbol_prefix_condition(symbol_column, codes)
        )
    )

In [20]:
# Build the IPC candidate query.
q_pool_ipc = query_pool(
    TLS209_APPLN_IPC,
    TLS209_APPLN_IPC.ipc_class_symbol,
    EXTERNAL_IPC_CODES,
)

# Build the CPC candidate query.
q_pool_cpc = query_pool(
    TLS224_APPLN_CPC,
    TLS224_APPLN_CPC.cpc_class_symbol,
    EXTERNAL_CPC_CODES,
)

# Combine IPC and CPC candidate IDs.
# UNION removes duplicate IDs across the two queries.
pool_ids = q_pool_ipc.union(q_pool_cpc).subquery()

# Keep only candidate IDs that are absent from the initial search union (keyword + IPC + CPC).
#pool_ids: ID della ricerca ampia appena costruita;
#drone_ids: ID dell’unione iniziale keyword + IPC + CPC.
external_ids = (
    db.query(pool_ids.c.appln_id)
    .select_from(pool_ids)
    .outerjoin(
        drone_ids,
        pool_ids.c.appln_id == drone_ids.c.appln_id,
    )
    .filter(drone_ids.c.appln_id.is_(None))
    .subquery()
)

In [ ]:
# Build the query for application metadata and available English texts.
q_external = query_details(external_ids)

# Execute the query and keep one row per application.
if USE_ENRICHMENT:
    df_external = one_application(
        patstat.df(q_external)
    )
else:
    df_external = df_drone.iloc[0:0].copy()     # enrichment off: empty pool, nothing to encode

print("Applications in the external pool:", len(df_external))

df_external.head(10)

In [22]:
# Select external applications with a non-empty English abstract.
df_external_sbert = df_external[
    df_external["appln_abstract"].fillna("").str.strip().ne("")
].copy()

print("External applications:", len(df_external))
print("Applications with an abstract:", len(df_external_sbert))

External applications: 7881
Applications with an abstract: 7136


In [23]:
# Encode external abstracts using the same model as the initial search.
df_external_sbert["sbert_similarity"] = np.nan

if not df_external_sbert.empty:
    external_embeddings = model.encode(
        df_external_sbert["appln_abstract"].tolist(),
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    # Compare each external abstract with the drone sector description.
    df_external_sbert["sbert_similarity"] = (
        external_embeddings @ drone_embedding
    )

Batches:   0%|          | 0/223 [00:00<?, ?it/s]

In [26]:
# Retain external applications that reach the enrichment threshold.
df_enrichment = df_external_sbert[
    df_external_sbert["sbert_similarity"] >= TAU_ENRICH
].copy()

print("External applications evaluated:", len(df_external_sbert))
print("Applications selected for enrichment:", len(df_enrichment))

df_enrichment[
    ["appln_id", "appln_title", "sbert_similarity", "appln_abstract"]
].sort_values("sbert_similarity", ascending=False).head(10)

# Non selezionati dal pool esterno
df_external_rejected = df_external_sbert[
    df_external_sbert["sbert_similarity"] < TAU_ENRICH
].copy()

External applications evaluated: 7136
Applications selected for enrichment: 1106


In [27]:
# Add a source label to distinguish applications retained from the initial
# keyword/IPC/CPC union from those selected through semantic enrichment.
df_filtered = df_filtered.copy()
df_filtered["retrieval_source"] = "initial_union"

df_enrichment["retrieval_source"] = "semantic_enrichment"

# Combine the SBERT-filtered initial results with the selected external results.
# This is a provisional dataset pending manual validation.
df_selected = pd.concat(
    [df_filtered, df_enrichment],
    ignore_index=True,
)

# Verify that each application appears only once in the combined dataset.
assert not df_selected["appln_id"].duplicated().any()

# Display the total number of selected applications.
print("Total selected applications:", len(df_selected))

Total selected applications: 3595


In [28]:
def split_validation_families(selected, rejected):
    """
    Prepara due gruppi disgiunti per la validazione.

    Una famiglia è selezionata se almeno una sua application
    supera la soglia. Le sue altre application non vengono
    considerate famiglie scartate.
    """
    def valid_families(df):
        result = df.copy()

        family_ids = pd.to_numeric(
            result["docdb_family_id"], errors="coerce"
        )

        valid = (
            family_ids.notna()
            & family_ids.gt(0)
            & family_ids.mod(1).eq(0)
        )

        result = result.loc[valid].copy()
        result["docdb_family_id"] = (
            family_ids.loc[valid].astype("int64")
        )

        return result

    selected = valid_families(selected)
    rejected = valid_families(rejected)

    selected_ids = set(selected["docdb_family_id"])

    # Una famiglia già recuperata non è un falso negativo,
    # anche se alcuni suoi membri sono sotto soglia.
    rejected = rejected.loc[
        ~rejected["docdb_family_id"].isin(selected_ids)
    ].copy()

    assert set(selected["docdb_family_id"]).isdisjoint(
        set(rejected["docdb_family_id"])
    )

    print("Famiglie selezionate:", selected["docdb_family_id"].nunique())
    print("Famiglie scartate:", rejected["docdb_family_id"].nunique())

    return selected, rejected


# 1. Valutazione del filtro iniziale.
print("FILTRO INIZIALE")

filter_selected_families, filter_rejected_families = (
    split_validation_families(df_filtered, df_rejected)
)

# Famiglie già recuperate dal filtro iniziale.
initial_selected_ids = set(
    filter_selected_families["docdb_family_id"]
)

# 2. Prepara gruppi disgiunti per l'arricchimento.
print("\nARRICCHIMENTO PRIMA DEL CONTROLLO SULLE FAMIGLIE")

enrichment_selected_families, enrichment_rejected_families = (
    split_validation_families(
        df_enrichment,
        df_external_rejected,
    )
)

# Conta le famiglie selezionate dall'arricchimento
# che erano già state recuperate dal filtro iniziale.
overlap_ids = initial_selected_ids.intersection(
    enrichment_selected_families["docdb_family_id"]
)

print("\nFamiglie già recuperate dal filtro:", len(overlap_ids))

# Per misurare il contributo aggiuntivo dell'arricchimento,
# escludi le famiglie già recuperate da entrambi i gruppi.
enrichment_selected_families = (
    enrichment_selected_families.loc[
        ~enrichment_selected_families["docdb_family_id"]
        .isin(initial_selected_ids)
    ].copy()
)

enrichment_rejected_families = (
    enrichment_rejected_families.loc[
        ~enrichment_rejected_families["docdb_family_id"]
        .isin(initial_selected_ids)
    ].copy()
)

new_family_ids = set(
    enrichment_selected_families["docdb_family_id"]
)

assert initial_selected_ids.isdisjoint(new_family_ids)

print(
    "Famiglie aggiunte dall'arricchimento:",
    len(new_family_ids),
)
print(
    "Famiglie nel pool esterno non recuperate da nessuno dei due passaggi:",
    enrichment_rejected_families["docdb_family_id"].nunique(),
)
print(
    "Famiglie selezionate complessivamente:",
    len(initial_selected_ids | new_family_ids),
)

FILTRO INIZIALE
Famiglie selezionate: 1970
Famiglie scartate: 314

ARRICCHIMENTO PRIMA DEL CONTROLLO SULLE FAMIGLIE
Famiglie selezionate: 923
Famiglie scartate: 4537

Famiglie già recuperate dal filtro: 1
Famiglie aggiunte dall'arricchimento: 922
Famiglie nel pool esterno non recuperate da nessuno dei due passaggi: 4537
Famiglie selezionate complessivamente: 2892


In [29]:
# Application senza abstract inglese: restano non valutate da SBERT.
def missing_abstract(df):
    return (
        df["appln_abstract"]
        .fillna("")
        .str.strip()
        .eq("")
    )


df_initial_no_abstract = df_drone.loc[
    missing_abstract(df_drone)
].copy()

df_external_no_abstract = df_external.loc[
    missing_abstract(df_external)
].copy()

df_initial_no_abstract["retrieval_source"] = "initial_union"
df_external_no_abstract["retrieval_source"] = "external_pool"

df_no_abstract = pd.concat(
    [df_initial_no_abstract, df_external_no_abstract],
    ignore_index=True,
)

df_no_abstract["evaluation_status"] = "not_evaluated_no_english_abstract"


# Estrae soltanto ID famiglia validi.
def valid_family_ids(df):
    ids = pd.to_numeric(df["docdb_family_id"], errors="coerce")
    valid = ids.notna() & ids.gt(0) & ids.mod(1).eq(0)
    return set(ids.loc[valid].astype("int64"))


missing_ids = valid_family_ids(df_no_abstract)

# Famiglie con almeno un membro con abstract nei dati estratti.
evaluated_ids = (
    valid_family_ids(df_sbert)
    | valid_family_ids(df_external_sbert)
)

selected_ids = valid_family_ids(df_selected)

# Tre gruppi disgiunti delle famiglie coinvolte nei testi mancanti.
recovered_ids = missing_ids & selected_ids

evaluated_not_selected_ids = (
    (missing_ids & evaluated_ids) - selected_ids
)

no_text_family_ids = missing_ids - evaluated_ids

assert len(missing_ids) == (
    len(recovered_ids)
    + len(evaluated_not_selected_ids)
    + len(no_text_family_ids)
)

print("Application iniziali senza abstract:", len(df_initial_no_abstract))
print("Application esterne senza abstract:", len(df_external_no_abstract))
print("Famiglie coinvolte:", len(missing_ids))

print(
    "Famiglie comunque selezionate tramite altri membri:",
    len(recovered_ids),
)
print(
    "Famiglie con altri membri valutati, ma non selezionate:",
    len(evaluated_not_selected_ids),
)
print(
    "Famiglie senza alcun abstract inglese nei dati estratti:",
    len(no_text_family_ids),
)

Application iniziali senza abstract: 232
Application esterne senza abstract: 745
Famiglie coinvolte: 636
Famiglie comunque selezionate tramite altri membri: 88
Famiglie con altri membri valutati, ma non selezionate: 260
Famiglie senza alcun abstract inglese nei dati estratti: 288


In [ ]:
# Una riga per famiglia coinvolta negli abstract mancanti.
family_text_status = pd.DataFrame({
    "docdb_family_id": sorted(missing_ids)
})

def text_status(family_id):
    if family_id in recovered_ids:
        return "selected_via_other_member"
    if family_id in evaluated_not_selected_ids:
        return "evaluated_other_member_not_selected"
    return "not_evaluated_no_english_abstract"

family_text_status["text_status"] = (
    family_text_status["docdb_family_id"].map(text_status)
)

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

df_no_abstract.to_csv(
    output_dir / "applications_without_english_abstract.csv",
    index=False,
)

family_text_status.to_csv(
    output_dir / "family_text_status.csv",
    index=False,
)

In [ ]:
#AUTOMATIC VALIDATION WIHT LLM
#SULLE APPLICATION SCARTATE E TENUTE DAL FILTRO SBERT
#SULLE APPLICATION SCARTATE E TENUTE DAL ENRICHMENT SBERT

# Esegue la configurazione del validatore e carica le sue funzioni.
# I due notebook devono trovarsi nella stessa cartella.
# -i permette al validatore di accedere alle variabili già definite,
# inclusa DRONE_DESCRIPTION.


%pip install -q langchain-core langchain-openai pydantic
%run -i ./2-DataValidation.ipynb
    
validation_kept = validate_sample(filter_selected_families)
validation_rejected = validate_sample(filter_rejected_families, n=100)



# I conteggi delle popolazioni devono usare gli stessi gruppi
# da cui sono stati estratti i campioni.
filter_metrics = evaluate_selection(
    validation_selected=validation_kept,
    validation_rejected=validation_rejected,
    df_selected=filter_selected_families,
    df_rejected=filter_rejected_families,
    name="Filtro SBERT",
)

rows = [filter_metrics]
if USE_ENRICHMENT:
    validation_added = validate_sample(enrichment_selected_families)
    validation_not_added = validate_sample(enrichment_rejected_families, n=100)
    enrichment_metrics = evaluate_selection(
        validation_selected=validation_added,
        validation_rejected=validation_not_added,
        df_selected=enrichment_selected_families,
        df_rejected=enrichment_rejected_families,
        name="Arricchimento SBERT",
    )
    rows.append(enrichment_metrics)

comparison = pd.DataFrame(rows)

display(comparison.round(3))



In [ ]:
from pathlib import Path

output_file = Path("/home/jovyan/progetto/df_selected.parquet")
output_file.parent.mkdir(parents=True, exist_ok=True)

df_selected.to_parquet(output_file, index=False)

print(f"Dataset saved to: {output_file}")
print(f"Applications: {len(df_selected):,}")
print(f"DOCDB families: {df_selected['docdb_family_id'].nunique():,}")